# Lab 6

You are tasked with evaluating card counting strategies for black jack. In order to do so, you will use object oriented programming to create a playable casino style black jack game where a computer dealer plays against $n$ computer players and possibly one human player. If you don't know the rules of blackjack or card counting, please google it. 

A few requirements:
* The game should utilize multiple 52-card decks. Typically the game is played with 6 decks.
* Players should have chips.
* Dealer's actions are predefined by rules of the game (typically hit on 16). 
* The players should be aware of all shown cards so that they can count cards.
* Each player could have a different strategy.
* The system should allow you to play large numbers of games, study the outcomes, and compare average winnings per hand rate for different strategies.

1. Begin by creating a classes to represent cards and decks. The deck should support more than one 52-card set. The deck should allow you to shuffle and draw cards. Include a "plastic" card, placed randomly in the deck. Later, when the plastic card is dealt, shuffle the cards before the next deal.

In [55]:
import random


# class with 52 cards, one extra plastic card, so 53 total
class Card:
  RANKS = ['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']
  SUITS = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
  show_suits = True

  def __init__(self, rank, suit, is_plastic=False):
    self.rank = rank
    self.suit = suit
    self.is_plastic = is_plastic

  def get_value(self):
    if self.is_plastic:
      return 0
    if self.rank in ['J', 'Q', 'K']:
      return 10
    if self.rank == 'A':
      return 11
    return int(self.rank)

  def __str__(self):
    if self.is_plastic:
      return 'PLASTIC CUT CARD'
    if Card.show_suits:
      return f'{self.rank} of {self.suit}'
    return f'{self.rank}'

  def __repr__(self):
    return self.__str__()


# class of decks of cards. can support more than one 52-set of cards. able to shuffle and draw cards. shuffles when plastic card is drawn before next deal
class Deck:

  def __init__(self, num_decks=6):
    self.num_decks = num_decks
    self.cards = []
    self.needs_shuffle = False
    # automatically build and shuffle the shoe when Deck is created
    self.shuffle_deck()

  def create_deck(self):
    self.cards = []
    for _ in range(self.num_decks):
      for suit in Card.SUITS:
        for rank in Card.RANKS:
          self.cards.append(Card(rank, suit))

  def shuffle_deck(self):
    self.create_deck()
    random.shuffle(self.cards)

    # insert plastic card at a random position in shoe, of course not top or bottom
    plastic_card = Card('PLASTIC', 'PLASTIC', is_plastic=True)
    insert_position = random.randint(1, len(self.cards) - 1)
    self.cards.insert(insert_position, plastic_card)

    self.needs_shuffle = False

  def draw_card(self):
    # if plastic card is drawn flags needs_shuffle for next deal and draws the next real card

    if len(self.cards) == 0:
      self.shuffle_deck()

    drawn_card = self.cards.pop()

    if drawn_card.is_plastic:
      self.needs_shuffle = True
      # draw the next card so a player never receives the plastic card
      return self.draw_card()

    return drawn_card

In [56]:
# test

random.seed(8)  # fixed seed for reproducible test output
deck = Deck(num_decks=6)
print(
    f'Created {deck.num_decks}-deck shoe with {len(deck.cards)} total'
    f' cards ({len(deck.cards)-1} regular + 1 plastic cut card)\n'
)

# preview the first 5 drawn cards and their values
print('First 5 cards drawn:')
for i in range(1, 6):
  sample_card = deck.draw_card()
  print(f'  Card {i}: {sample_card} (Value: {sample_card.get_value()})')
print()

# reset shoe and simulate dealing 80 four-card rounds (320 cards total)
deck.shuffle_deck()
total_cards_dealt = 0
plastic_triggered_count = 0

for round_num in range(1, 81):
  # check before each new deal if the plastic card was drawn in the previous round
  if deck.needs_shuffle:
    print(
        f'[Before Round {round_num}] Reshuffling shoe because plastic card'
        ' was reached in the previous round'
    )
    deck.shuffle_deck()
    print(f'  -> Shoe restored to {len(deck.cards)} cards\n')

  # simulate dealing 4 cards in this round
  round_cards = []
  for _ in range(4):
    was_flagged = deck.needs_shuffle
    card = deck.draw_card()
    round_cards.append(card)
    total_cards_dealt += 1

    # verify that the plastic card was never dealt into a hand
    assert not card.is_plastic, 'Error: Plastic card was dealt to a player!'
    assert 2 <= card.get_value() <= 11, f'Error: Invalid card value on {card}'

    if not was_flagged and deck.needs_shuffle:
      plastic_triggered_count += 1
      print(
          f'[Round {round_num}] Plastic cut card encountered mid-round!'
          f' Dealt replacement card: {card}'
      )
      print(
          f'  -> Cards remaining in shoe: {len(deck.cards)} |'
          f' needs_shuffle: {deck.needs_shuffle}'
      )

print(
    f'\nTest complete: {total_cards_dealt} real cards dealt across 80 rounds,'
    f' plastic cut card triggered {plastic_triggered_count} time(s), and 0'
    ' plastic cards ended up in a hand'
)

Created 6-deck shoe with 313 total cards (312 regular + 1 plastic cut card)

First 5 cards drawn:
  Card 1: A of Hearts (Value: 11)
  Card 2: 9 of Clubs (Value: 9)
  Card 3: Q of Clubs (Value: 10)
  Card 4: A of Hearts (Value: 11)
  Card 5: 9 of Spades (Value: 9)

[Round 61] Plastic cut card encountered mid-round! Dealt replacement card: 6 of Clubs
  -> Cards remaining in shoe: 70 | needs_shuffle: True
[Before Round 62] Reshuffling shoe because plastic card was reached in the previous round
  -> Shoe restored to 313 cards


Test complete: 320 real cards dealt across 80 rounds, plastic cut card triggered 1 time(s), and 0 plastic cards ended up in a hand


2. Now design your game on a UML diagram. You may want to create classes to represent, players, a hand, and/or the game. As you work through the lab, update your UML diagram. At the end of the lab, submit your diagram (as pdf file) along with your notebook. 

3. Begin with implementing the skeleton (ie define data members and methods/functions, but do not code the logic) of the classes in your UML diagram.

In [29]:
class Hand:

  def __init__(self):
    self.cards = []
    self.bet = 0

  def add_card(self, card):
    pass

  def get_total(self):
    pass

  def is_bust(self):
    pass

  def is_blackjack(self):
    pass

  def clear(self):
    pass


class Player:

  def __init__(self, name='Player', chips=1000):
    self.name = name
    self.chips = chips
    self.hand = Hand()
    self.hands_played = 0
    self.net_winnings = 0

  def place_bet(self, seen_cards):
    pass

  def decide_action(self, dealer_upcard, seen_cards):
    pass

  def win_bet(self, multiplier):
    pass


class Dealer(Player):

  def __init__(self):
    super().__init__(name='Dealer', chips=0)

  def decide_action(self, dealer_upcard, seen_cards):
    pass


class HumanPlayer(Player):

  def __init__(self, name='Human', chips=1000):
    super().__init__(name=name, chips=chips)

  def place_bet(self, seen_cards):
    pass

  def decide_action(self, dealer_upcard, seen_cards):
    pass


class AIPlayer(Player):

  def __init__(self, name='AI', chips=1000, strategy_type='basic'):
    super().__init__(name=name, chips=chips)
    self.strategy_type = strategy_type

  def place_bet(self, seen_cards):
    pass

  def decide_action(self, dealer_upcard, seen_cards):
    pass

  def count_cards(self, seen_cards):
    pass


class BlackjackGame:

  def __init__(self, players=None, num_decks=6):
    self.deck = Deck(num_decks=num_decks)
    self.dealer = Dealer()
    self.players = players if players is not None else []
    self.seen_cards = []

  def deal_initial_cards(self):
    pass

  def play_round(self):
    pass

  def check_win(self):
    pass

  def display(self):
    pass

  def run_simulation(self, num_games):
    pass

4. Complete the implementation by coding the logic of all functions. For now, just implement the dealer player and human player.

In [60]:
class Hand:

  def __init__(self):
    self.cards = []
    self.bet = 0

  # adds dealt card to hand
  def add_card(self, card):
    self.cards.append(card)

  # calculate hand total value. if ace causes bust, reduce to 1 from 11
  def get_total(self):
    total = sum(card.get_value() for card in self.cards)
    aces = sum(1 for card in self.cards if card.rank == 'A')
    while total > 21 and aces > 0:
      total -= 10
      aces -= 1
    return total

  # returns true if hand over 21
  def is_bust(self):
    return self.get_total() > 21

  # returns true if hand is a natural 2 card 21
  def is_blackjack(self):
    return len(self.cards) == 2 and self.get_total() == 21

  # resets hand and bet before new round
  def clear(self):
    self.cards = []
    self.bet = 0

  def __str__(self):
    cards_str = ', '.join(str(card) for card in self.cards)
    return f'[{cards_str}] (Total: {self.get_total()})'


class Player:

  def __init__(self, name='Player', chips=1000):
    self.name = name
    self.chips = chips
    self.hand = Hand()
    self.hands_played = 0
    self.net_winnings = 0

  # reduces chips by bet amount and attaches to hand
  def place_bet(self, seen_cards, amount=10):
    bet = min(amount, self.chips)
    self.chips -= bet
    self.net_winnings -= bet
    self.hand.bet = bet
    return bet

  # default player actions (hit or stand)
  def decide_action(self, dealer_upcard, seen_cards):
    return 'stand'

  # pays out chips based on multiplier (0=loss, 1=push, 2=win, 2.5=blackjack)
  def win_bet(self, multiplier):
    payout = int(self.hand.bet * multiplier)
    self.chips += payout
    self.net_winnings += payout


class Dealer(Player):

  def __init__(self, name='Dealer', chips=0):
    super().__init__(name=name, chips=chips)

  def decide_action(self, dealer_upcard=None, seen_cards=None):
    # dealer must hit on 16 or under, and stand on 17 or over
    if self.hand.get_total() <= 16:
      return 'hit'
    return 'stand'


class HumanPlayer(Player):

  def __init__(self, name='Human', chips=1000):
    super().__init__(name=name, chips=chips)

  def place_bet(self, seen_cards):
    # prompts the human player to enter a valid chip bet
    while True:
      try:
        raw = input(
            f'\n{self.name}, you have {self.chips} chips. Enter your bet: '
        ).strip()
        bet = int(raw)
        if 1 <= bet <= self.chips:
          self.chips -= bet
          self.net_winnings -= bet
          self.hand.bet = bet
          return bet
        print(f'Please enter a bet between 1 and {self.chips}.')
      except ValueError:
        print('Invalid input. Please enter a whole number.')

  # prompts human player to hit or stand
  def decide_action(self, dealer_upcard, seen_cards):
    while True:
      choice = (
          input(
              f'{self.name} ({self.hand}) vs Dealer showing [{dealer_upcard}]'
              ' -> (h)it or (s)tand? '
          )
          .strip()
          .lower()
      )
      if choice in ['h', 'hit']:
        return 'hit'
      if choice in ['s', 'stand']:
        return 'stand'
      print("Invalid choice. Please type 'h' to hit or 's' to stand.")


class BlackjackGame:

  def __init__(self, players=None, num_decks=6, verbose=True, show_suits=True):
    self.deck = Deck(num_decks=num_decks)
    self.dealer = Dealer()
    self.players = players if players is not None else [HumanPlayer()]
    self.seen_cards = []
    Card.show_suits = show_suits
    self.verbose = verbose

  # draws a card from the shoe and adds it to "seen_cards" if faceup
  def _draw_card(self, reveal=True):
    card = self.deck.draw_card()
    if reveal:
      self.seen_cards.append(card)
    return card

  # deals 2 cards to each active player and the dealer (dealer's 2nd card is facedown)
  def deal_initial_cards(self):
    for round_idx in range(2):
      for player in self.players:
        if player.hand.bet > 0:
          player.hand.add_card(self._draw_card(reveal=True))
      # dealer's first card is faceup. second card is facedown
      is_upcard = round_idx == 0
      self.dealer.hand.add_card(self._draw_card(reveal=is_upcard))

  # prints the current table state
  def display(self, hide_dealer_hole_card=True):
    if not self.verbose:
      return
    print('\n--- Current Table ---')
    if hide_dealer_hole_card and len(self.dealer.hand.cards) >= 2:
      upcard = self.dealer.hand.cards[0]
      print(f'Dealer: [{upcard}, HIDDEN] (Showing: {upcard.get_value()})')
    else:
      print(f'Dealer: {self.dealer.hand}')

    for player in self.players:
      if player.hand.bet > 0:
        print(
            f'{player.name}: {player.hand} | Bet: {player.hand.bet} | Chips'
            f' remaining: {player.chips}'
        )
    print(f'Seen cards in shoe so far: {len(self.seen_cards)}')

  # executes one complete round of Blackjack
  def play_round(self):
    # 1. reshuffle before the deal if the plastic card was drawn last round
    if self.deck.needs_shuffle:
      if self.verbose:
        print('\n[Shoe] Plastic card was reached! Reshuffling shoe...')
      self.deck.shuffle_deck()
      self.seen_cards.clear()

    # 2. clear previous hands and collect bets from players who have chips
    self.dealer.hand.clear()
    active_players = []
    for player in self.players:
      player.hand.clear()
      if player.chips > 0:
        player.place_bet(self.seen_cards)
        active_players.append(player)
      elif self.verbose:
        print(f'{player.name} is out of chips and sits out this round.')

    if not active_players:
      if self.verbose:
        print('No active players with chips remaining.')
      return

    # 3. deal initial 2 cards and display the opening table
    self.deal_initial_cards()
    self.display(hide_dealer_hole_card=True)

    dealer_upcard = self.dealer.hand.cards[0]

    # 4. each player takes their turn (unless dealer already has a natural Blackjack)
    if not self.dealer.hand.is_blackjack():
      for player in active_players:
        if player.hand.is_blackjack():
          if self.verbose:
            print(f'{player.name} has a natural Blackjack!')
          continue

        while not player.hand.is_bust():
          action = player.decide_action(dealer_upcard, self.seen_cards)
          if action == 'hit':
            new_card = self._draw_card(reveal=True)
            player.hand.add_card(new_card)
            if self.verbose:
              print(
                  f'  -> {player.name} hits and draws {new_card}. Hand:'
                  f' {player.hand}'
              )
            if player.hand.is_bust() and self.verbose:
              print(f'  -> {player.name} busts with {player.hand.get_total()}!')
          else:
            if self.verbose:
              print(f'  -> {player.name} stands at {player.hand.get_total()}.')
            break

    # 5. reveal dealer's hole card so card counters can track it
    if len(self.dealer.hand.cards) >= 2:
      hole_card = self.dealer.hand.cards[1]
      self.seen_cards.append(hole_card)
      if self.verbose:
        print(f'\nDealer reveals hole card: {hole_card} -> {self.dealer.hand}')

    # dealer plays only if at least one player is still in the hand
    players_needing_dealer = [
        p
        for p in active_players
        if not p.hand.is_bust() and not p.hand.is_blackjack()
    ]
    if players_needing_dealer and not self.dealer.hand.is_blackjack():
      while not self.dealer.hand.is_bust():
        action = self.dealer.decide_action(dealer_upcard, self.seen_cards)
        if action == 'hit':
          new_card = self._draw_card(reveal=True)
          self.dealer.hand.add_card(new_card)
          if self.verbose:
            print(
                f'  -> Dealer hits and draws {new_card}. Hand:'
                f' {self.dealer.hand}'
            )
          if self.dealer.hand.is_bust() and self.verbose:
            print(f'  -> Dealer busts with {self.dealer.hand.get_total()}!')
        else:
          if self.verbose:
            print(f'  -> Dealer stands at {self.dealer.hand.get_total()}.')
          break

    # 6. settle bets and record statistics
    self.check_win()

  # compares each player's hand against the dealer's and settles bets
  def check_win(self):
    dealer_total = self.dealer.hand.get_total()
    dealer_bust = self.dealer.hand.is_bust()
    dealer_bj = self.dealer.hand.is_blackjack()

    if self.verbose:
      print('\n--- Round Results ---')

    for player in self.players:
      if player.hand.bet == 0:
        continue

      player.hands_played += 1
      player_total = player.hand.get_total()
      player_bust = player.hand.is_bust()
      player_bj = player.hand.is_blackjack()

      if player_bust:
        player.win_bet(0.0)
        outcome = f'LOST {player.hand.bet} chips (Busted)'
      elif player_bj and dealer_bj:
        player.win_bet(1.0)
        outcome = 'PUSH (Both have Blackjack)'
      elif player_bj:
        player.win_bet(2.5)  # 3:2 payout on natural Blackjack
        profit = int(player.hand.bet * 1.5)
        outcome = f'BLACKJACK! Won +{profit} chips (3:2 payout)'
      elif dealer_bj:
        player.win_bet(0.0)
        outcome = f'LOST {player.hand.bet} chips (Dealer Blackjack)'
      elif dealer_bust or player_total > dealer_total:
        player.win_bet(2.0)
        outcome = (
            f'WON +{player.hand.bet} chips ({player_total} vs {dealer_total})'
        )
      elif player_total == dealer_total:
        player.win_bet(1.0)
        outcome = f'PUSH at {player_total}'
      else:
        player.win_bet(0.0)
        outcome = (
            f'LOST {player.hand.bet} chips ({player_total} vs {dealer_total})'
        )

      if self.verbose:
        print(
            f'{player.name}: {outcome} | Chips: {player.chips} | Net:'
            f' {player.net_winnings:+d}'
        )

  # runs multiple rounds of Blackjack and returns summary statistics for each player
  def run_simulation(self, num_games):
    for _ in range(num_games):
      self.play_round()

    stats = {}
    for player in self.players:
      avg_rate = (
          player.net_winnings / player.hands_played
          if player.hands_played > 0
          else 0.0
      )
      stats[player.name] = {
          'hands_played': player.hands_played,
          'final_chips': player.chips,
          'net_winnings': player.net_winnings,
          'avg_winnings_per_hand': round(avg_rate, 4),
      }
    return stats

In [65]:
# human test

me = HumanPlayer(name='human', chips=1000)
interactive_game = BlackjackGame(players=[me], num_decks=6, verbose=True, show_suits=False)

while me.chips > 0:
  interactive_game.play_round()

  if me.chips <= 0:
    print('\nYou are out of chips! Game over.')
    break

  keep_playing = input('\nPlay another hand? (y/n): ').strip().lower()
  if keep_playing not in ['y', 'yes']:
    break

print(
    f'\nFinal Stats -> Hands played: {me.hands_played} | Final chips:'
    f' {me.chips} | Net winnings: {me.net_winnings:+d}'
)


--- Current Table ---
Dealer: [7, HIDDEN] (Showing: 7)
human: [6, 8] (Total: 14) | Bet: 800 | Chips remaining: 200
Seen cards in shoe so far: 3
  -> human hits and draws J. Hand: [6, 8, J] (Total: 24)
  -> human busts with 24!

Dealer reveals hole card: 10 -> [7, 10] (Total: 17)

--- Round Results ---
human: LOST 800 chips (Busted) | Chips: 200 | Net: -800

Final Stats -> Hands played: 1 | Final chips: 200 | Net winnings: -800


5.  Test. Demonstrate game play. For example, create a game of several dealer players and show that the game is functional through several rounds.

In [ ]:
# simulated test

random.seed(8)

dealer_players = [
    Dealer(name='DealerPlayer 1', chips=1000),
    Dealer(name='DealerPlayer 2', chips=1000),
    Dealer(name='DealerPlayer 3', chips=1000),
]

game_step5 = BlackjackGame(players=dealer_players, num_decks=6, verbose=True)
stats_step5 = game_step5.run_simulation(num_games=5)

print('\n=== Step 5 Summary After 5 Rounds ===')
for player_name, player_stats in stats_step5.items():
  print(f'{player_name}: {player_stats}')



--- Current Table ---
Dealer: [A of Hearts, HIDDEN] (Showing: 11)
DealerPlayer 1: [A of Hearts, 9 of Spades] (Total: 20) | Bet: 10 | Chips remaining: 990
DealerPlayer 2: [9 of Clubs, J of Diamonds] (Total: 19) | Bet: 10 | Chips remaining: 990
DealerPlayer 3: [Q of Clubs, 6 of Spades] (Total: 16) | Bet: 10 | Chips remaining: 990
Seen cards in shoe so far: 7
  -> DealerPlayer 1 stands at 20.
  -> DealerPlayer 2 stands at 19.
  -> DealerPlayer 3 hits and draws J of Diamonds. Hand: [Q of Clubs, 6 of Spades, J of Diamonds] (Total: 26)
  -> DealerPlayer 3 busts with 26!

Dealer reveals hole card: 7 of Diamonds -> [A of Hearts, 7 of Diamonds] (Total: 18)
  -> Dealer stands at 18.

--- Round Results ---
DealerPlayer 1: WON +10 chips (20 vs 18) | Chips: 1010 | Net: +10
DealerPlayer 2: WON +10 chips (19 vs 18) | Chips: 1010 | Net: +10
DealerPlayer 3: LOST 10 chips (Busted) | Chips: 990 | Net: -10

--- Current Table ---
Dealer: [4 of Diamonds, HIDDEN] (Showing: 4)
DealerPlayer 1: [A of Spades, 3

6. Implement a new player with the following strategy:

    * Assign each card a value: 
        * Cards 2 to 6 are +1 
        * Cards 7 to 9 are 0 
        * Cards 10 through Ace are -1
    * Compute the sum of the values for all cards seen so far.
    * Hit if sum is very negative, stay if sum is very positive. Select a threshold for hit/stay, e.g. 0 or -2.  

In [37]:
class AIPlayer(Player):

  def __init__(self, name='CardCounter', chips=1000, threshold=-2):
    super().__init__(name=name, chips=chips)
    self.threshold = threshold

  # assign +1 to 2-6, 0 to 7-9, and -1 to 10-Ace for all seen cards
  def count_cards(self, seen_cards):
    count = 0
    for card in seen_cards:
      if card.rank in ['2', '3', '4', '5', '6']:
        count += 1
      elif card.rank in ['7', '8', '9']:
        count += 0
      elif card.rank in ['10', 'J', 'Q', 'K', 'A']:
        count -= 1
    return count

  # hit if running count is <= threshold (negative), stand if > threshold (positive)
  def decide_action(self, dealer_upcard, seen_cards):
    running_count = self.count_cards(seen_cards)
    if running_count <= self.threshold:
      return 'hit'
    return 'stand'


# quick test of AIPlayer over 3 rounds with threshold = -2
random.seed(42)
counter_player = AIPlayer(
    name='Counter (threshold=-2)', chips=1000, threshold=-2
)
game_step6 = BlackjackGame(players=[counter_player], num_decks=6, verbose=True)
stats_step6 = game_step6.run_simulation(num_games=3)

print('\n=== Step 6 Card Counter Demo Summary ===')
for player_name, player_stats in stats_step6.items():
  print(f'{player_name}: {player_stats}')


--- Current Table ---
Dealer: [A of Hearts, HIDDEN] (Showing: 11)
Counter (threshold=-2): [7 of Hearts, Q of Clubs] (Total: 17) | Bet: 10 | Chips remaining: 990
Seen cards in shoe so far: 3

Dealer reveals hole card: 10 of Diamonds -> [A of Hearts, 10 of Diamonds] (Total: 21)

--- Round Results ---
Counter (threshold=-2): LOST 10 chips (Dealer Blackjack) | Chips: 990 | Net: -10

--- Current Table ---
Dealer: [8 of Diamonds, HIDDEN] (Showing: 8)
Counter (threshold=-2): [Q of Hearts, 2 of Hearts] (Total: 12) | Bet: 10 | Chips remaining: 980
Seen cards in shoe so far: 7
  -> Counter (threshold=-2) hits and draws 7 of Spades. Hand: [Q of Hearts, 2 of Hearts, 7 of Spades] (Total: 19)
  -> Counter (threshold=-2) hits and draws 5 of Spades. Hand: [Q of Hearts, 2 of Hearts, 7 of Spades, 5 of Spades] (Total: 24)
  -> Counter (threshold=-2) busts with 24!

Dealer reveals hole card: 8 of Diamonds -> [8 of Diamonds, 8 of Diamonds] (Total: 16)

--- Round Results ---
Counter (threshold=-2): LOST 10

7. Create a test scenario where one player, using the above strategy, is playing with a dealer and 3 other players that follow the dealer's strategy. Each player starts with same number of chips. Play 50 rounds (or until the strategy player is out of money). Compute the strategy player's winnings. You may remove unnecessary printouts from your code (perhaps implement a verbose/quiet mode) to reduce the output.

8. Create a loop that runs 100 games of 50 rounds, as setup in previous question, and store the strategy player's chips at the end of the game (aka "winnings") in a list. Histogram the winnings. What is the average winnings per round? What is the standard deviation. What is the probability of net winning or losing after 50 rounds?


9. Repeat previous questions scanning the value of the threshold. Try at least 5 different threshold values. Can you find an optimal value?

10. Create a new strategy based on web searches or your own ideas. Demonstrate that the new strategy will result in increased or decreased winnings. 